# 🧊 TRELLIS — Image to 3D (Official Method, Colab-Ready)

> **Microsoft TRELLIS** — แปลงรูปภาพเป็น 3D Model (.glb / .ply)  
> [GitHub Official](https://github.com/microsoft/TRELLIS) | [HuggingFace Demo](https://huggingface.co/spaces/Microsoft/TRELLIS) | [Paper CVPR'25](https://arxiv.org/abs/2412.01506)

---
## ⚠️ อ่านก่อนรัน!

### สิ่งที่ต้องทำก่อน:
1. **Runtime → Change runtime type → T4 GPU** (ต้องการ VRAM ≥ 16GB)
2. กด **Runtime → Run all**
3. Cell 1 (condacolab) จะ **restart kernel อัตโนมัติ** — รอแล้วกด **Run all อีกครั้ง** ปกติมากครับ!
4. รอ build เสร็จ ~20-35 นาที ครั้งแรก
5. คลิก Gradio link ที่ขึ้นมา → อัปโหลดรูป → Generate!

---
### ⚡ สิ่งที่แก้ไขจาก Official setup.sh:
| ประเด็น | Official | Notebook นี้ |
|--------|----------|-------------|
| Python version | 3.10 | 3.10 (ตรงตาม official) |
| PyTorch | 2.4.0 | 2.4.0 (ตรงตาม official) |
| CUDA | 11.8 | 11.8 |
| condacolab limitation | ห้ามสร้าง env ใหม่ | ใช้ base env แทน |
| setup.sh flags | --new-env ไม่ได้ใน Colab | ข้ามและทำเอง |


## 🐍 Cell 1 — ติดตั้ง condacolab
> ⚠️ **Kernel จะ restart หลัง cell นี้ — นั่นคือปกติ!**  
> หลัง restart ให้กด **Runtime → Run all** ใหม่อีกครั้ง  
> Cell นี้จะ detect ว่าติดตั้งแล้วหรือยัง ถ้าติดตั้งแล้วจะข้ามไปเอง

In [ ]:
import subprocess, sys

# ตรวจสอบว่า condacolab ติดตั้งแล้วหรือยัง
def is_conda_installed():
    result = subprocess.run(['conda', '--version'], capture_output=True, text=True)
    return result.returncode == 0

if is_conda_installed():
    print('✅ Conda already installed! Skipping condacolab installation.')
    print('   Conda version:', subprocess.check_output(['conda', '--version']).decode().strip())
else:
    print('=' * 60)
    print('🐍 Installing condacolab (Miniconda on Colab)...')
    print('   ⚠️  Kernel will restart automatically — this is NORMAL!')
    print('   After restart, click Runtime → Run all again.')
    print('=' * 60)
    !pip install -q condacolab
    import condacolab
    condacolab.install_miniconda()
    # Kernel restarts here automatically

## ✅ Cell 2 — ตรวจสอบ Conda + GPU

In [ ]:
import subprocess

print('=' * 60)
print('✅ Environment Check')
print('=' * 60)

# ตรวจ conda
result = subprocess.run(['conda', '--version'], capture_output=True, text=True)
if result.returncode == 0:
    print(f'🐍 Conda: {result.stdout.strip()}')
else:
    print('❌ Conda NOT found! Please run Cell 1 first.')
    raise RuntimeError('Conda not installed')

# ตรวจ GPU
print('\n🖥️  GPU:')
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader

# ตรวจ disk
print('\n📀 Disk:')
!df -h /content | tail -1

print('\n✅ Ready to proceed!')

## ⚡ Cell 3 — ติดตั้ง CUDA 11.8
> Official TRELLIS ต้องการ CUDA 11.8 สำหรับ compile submodules

In [ ]:
%%bash
set -e

# ตรวจสอบว่า CUDA 11.8 ติดตั้งแล้วหรือยัง
if [ -f /usr/local/cuda-11.8/bin/nvcc ]; then
    echo '✅ CUDA 11.8 already installed!'
    /usr/local/cuda-11.8/bin/nvcc --version
    exit 0
fi

echo '========================================'
echo '⚡ Installing CUDA 11.8...'
echo '========================================'

apt-get update -y -qq
apt-get install -y -qq gnupg wget

wget -q https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64/cuda-keyring_1.1-1_all.deb
dpkg -i cuda-keyring_1.1-1_all.deb
apt-get update -y -qq
apt-get install -y -qq cuda-11-8

export PATH="/usr/local/cuda-11.8/bin:$PATH"
echo '✅ CUDA 11.8 installed!'
nvcc --version | head -4

## 📦 Cell 4 — ติดตั้ง PyTorch 2.4.0 + Dependencies
> ใช้ **base conda env** (condacolab ห้าม create env ใหม่)  
> Version ตรงตาม Official setup.sh: PyTorch 2.4.0 + CUDA 11.8

In [ ]:
%%bash
set -e

echo '========================================'
echo '📦 Installing PyTorch 2.4.0 + CUDA 11.8'
echo '   (Official TRELLIS version)'
echo '========================================'

# ตรวจสอบว่าติดตั้งแล้วหรือยัง
TORCH_VER=$(python -c "import torch; print(torch.__version__)" 2>/dev/null || echo "not_installed")
if [[ "$TORCH_VER" == *"2.4.0"* ]]; then
    echo "✅ PyTorch $TORCH_VER already installed! Skipping..."
else
    echo "Current PyTorch: $TORCH_VER — installing 2.4.0..."
    conda install -y \
        pytorch==2.4.0 \
        torchvision==0.19.0 \
        pytorch-cuda=11.8 \
        -c pytorch -c nvidia -q
    echo '✅ PyTorch 2.4.0 installed!'
fi

# ตรวจสอบ
python -c "
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'CUDA version: {torch.version.cuda}')
    print(f'GPU: {torch.cuda.get_device_name(0)}')
"

## 🔧 Cell 5 — ติดตั้ง System Build Tools

In [ ]:
%%bash
set -e

echo '========================================'
echo '🔧 Installing system build tools...'
echo '========================================'

apt-get update -y -qq
apt-get install -y -qq \
    build-essential \
    cmake \
    ninja-build \
    libgl1-mesa-dev \
    libglib2.0-0 \
    libsm6 libxext6 libxrender-dev libgomp1

# อัปเกรด pip tools
pip install -q --upgrade pip setuptools wheel ninja Cython

echo '✅ Build tools ready!'
gcc --version | head -1
cmake --version | head -1

## 📂 Cell 6 — Clone TRELLIS (Official Microsoft Repo)

In [ ]:
%%bash
set -e

# ตรวจสอบว่า clone แล้วหรือยัง
if [ -d /content/TRELLIS ] && [ -f /content/TRELLIS/setup.sh ]; then
    echo '✅ TRELLIS already cloned! Skipping...'
    echo 'Files in /content/TRELLIS:'
    ls /content/TRELLIS/
    exit 0
fi

echo '========================================'
echo '📂 Cloning microsoft/TRELLIS...'
echo '========================================'

rm -rf /content/TRELLIS

git clone --recurse-submodules \
    https://github.com/microsoft/TRELLIS.git \
    /content/TRELLIS

cd /content/TRELLIS
git submodule update --init --recursive

# สร้างโฟลเดอร์ที่ app.py ต้องการ
mkdir -p /content/TRELLIS/assets/example_image
mkdir -p /content/outputs

echo ''
echo '✅ TRELLIS cloned!'
ls /content/TRELLIS/

## 🏗️ Cell 7 — รัน Official setup.sh
> ใช้ flags ตรงจาก Official README  
> ขั้นตอนนี้ใช้เวลา **15–30 นาที** — อย่ากด interrupt!

In [ ]:
%%bash
set -e

echo '========================================'
echo '🏗️  Running official setup.sh...'
echo '    FLAGS: --basic --xformers --flash-attn'
echo '           --diffoctreerast --spconv'
echo '           --mipgaussian --kaolin --nvdiffrast --demo'
echo '    เวลา: ~15-30 นาที'
echo '========================================'

# ตั้งค่า CUDA PATH
export PATH="/usr/local/cuda-11.8/bin:$PATH"
export LD_LIBRARY_PATH="/usr/local/cuda-11.8/lib64:$LD_LIBRARY_PATH"
export CUDA_HOME="/usr/local/cuda-11.8"
export TORCH_CUDA_ARCH_LIST="7.0;7.5;8.0;8.6+PTX"

cd /content/TRELLIS

# รัน setup.sh โดยไม่ใช้ --new-env (เพราะ condacolab ใช้ base env)
# ใช้ . (source) เพื่อ activate conda environment
. ./setup.sh \
    --basic \
    --xformers \
    --flash-attn \
    --diffoctreerast \
    --spconv \
    --mipgaussian \
    --kaolin \
    --nvdiffrast \
    --demo

echo ''
echo '✅ setup.sh complete!'

## 🔍 Cell 8 — ตรวจสอบว่าติดตั้งสำเร็จ

In [ ]:
import sys
sys.path.insert(0, '/content/TRELLIS')

import os
os.environ['SPCONV_ALGO'] = 'native'
os.environ['OPENCV_IO_ENABLE_OPENEXR'] = '1'

print('=' * 60)
print('🔍 Verifying TRELLIS installation...')
print('=' * 60)

checks = {}

# PyTorch + CUDA
try:
    import torch
    checks['torch'] = f'{torch.__version__} (CUDA: {torch.cuda.is_available()})'
except Exception as e:
    checks['torch'] = f'❌ {e}'

# Flash Attention
try:
    import flash_attn
    checks['flash_attn'] = f'✅ {flash_attn.__version__}'
except:
    checks['flash_attn'] = '⚠️ Not installed (will use xformers fallback)'

# xformers
try:
    import xformers
    checks['xformers'] = f'✅ {xformers.__version__}'
except:
    checks['xformers'] = '❌ Not installed'

# spconv
try:
    import spconv
    checks['spconv'] = '✅ OK'
except:
    checks['spconv'] = '❌ Not installed'

# TRELLIS pipeline
try:
    from trellis.pipelines import TrellisImageTo3DPipeline
    checks['trellis'] = '✅ Pipeline importable'
except Exception as e:
    checks['trellis'] = f'❌ {e}'

# Gradio
try:
    import gradio as gr
    checks['gradio'] = f'✅ {gr.__version__}'
except:
    checks['gradio'] = '❌ Not installed'

for pkg, status in checks.items():
    icon = '✅' if '✅' in str(status) else ('⚠️' if '⚠️' in str(status) else '❌')
    print(f'  {icon} {pkg:15}: {status}')

all_ok = all('❌' not in str(v) for v in checks.values())
print()
if all_ok:
    print('🎉 All checks passed! Ready to run TRELLIS.')
else:
    print('⚠️  Some packages missing — check errors above before proceeding.')

## 🎨 Cell 9 — Patch app.py + Launch Gradio UI
> Patch ให้รัน `share=True` แล้วเปิด Public URL
>
> 🕐 **ครั้งแรก**: รอ download model weights ~2-5 นาที (ประมาณ 6-8 GB จาก HuggingFace)  
> 🔗 **URL**: จะขึ้น `Running on public URL: https://xxxx.gradio.live`

In [ ]:
import os, subprocess, sys

APP_PY = '/content/TRELLIS/app.py'

# ---- Patch app.py ----
print('🔧 Patching app.py for Colab...')
with open(APP_PY, 'r') as f:
    code = f.read()

original = code  # เก็บไว้เปรียบเทียบ

# Patch 1: share=True
for old, new in [
    ('demo.launch()', 'demo.launch(share=True, server_port=7860, server_name="0.0.0.0")'),
    ('demo.launch(share=False)', 'demo.launch(share=True, server_port=7860, server_name="0.0.0.0")'),
    ('demo.queue().launch()', 'demo.queue().launch(share=True, server_port=7860, server_name="0.0.0.0")'),
]:
    if old in code:
        code = code.replace(old, new)
        print(f'  ✅ Patched: {old!r}')

if code == original:
    print('  ⚠️  No launch() found to patch — checking app.py manually')
    # หา launch line
    for i, line in enumerate(original.split('\n')):
        if 'launch' in line:
            print(f'    Line {i}: {line.strip()}')

with open(APP_PY, 'w') as f:
    f.write(code)

print('\n✅ app.py patched!')
print('\n' + '=' * 60)
print('🚀 Launching TRELLIS Gradio App...')
print('=' * 60)
print('📌 รอ URL: "Running on public URL: https://xxxx.gradio.live"')
print('🕐 Download model weights ครั้งแรก ~2-5 นาที (6-8 GB)')
print()

# ตั้งค่า environment
env = os.environ.copy()
env.update({
    'PATH': '/usr/local/cuda-11.8/bin:' + env.get('PATH', ''),
    'LD_LIBRARY_PATH': '/usr/local/cuda-11.8/lib64:' + env.get('LD_LIBRARY_PATH', ''),
    'CUDA_HOME': '/usr/local/cuda-11.8',
    'SPCONV_ALGO': 'native',
    'OPENCV_IO_ENABLE_OPENEXR': '1',
    'PYTORCH_CUDA_ALLOC_CONF': 'expandable_segments:True',
})

proc = subprocess.Popen(
    [sys.executable, '-u', 'app.py'],
    cwd='/content/TRELLIS',
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    env=env
)

try:
    for line in iter(proc.stdout.readline, b''):
        text = line.decode('utf-8', errors='replace').rstrip()
        print(text)
        if 'gradio.live' in text or 'Running on public URL' in text:
            print()
            print('🎉' * 20)
            print('🔗 คลิก URL ด้านบนเพื่อเปิด TRELLIS UI!')
            print('🎉' * 20)
            print()
except KeyboardInterrupt:
    print('\n⛔ Stopped by user.')
    proc.kill()
finally:
    proc.wait()

---
## 🐍 Optional — Inference โดยตรง (ไม่ใช้ UI)
> อัปโหลดรูปผ่าน Files panel ซ้ายมือ แล้วใส่ path ด้านล่าง

In [ ]:
# ==================================================
IMAGE_PATH  = "/content/my_image.png"  # ← ใส่ path รูป
OUTPUT_PATH = "/content/output.glb"   # ← path output
SEED        = 42
STEPS       = 12                       # 12-25 ยิ่งมากยิ่งละเอียด
# ==================================================

import os, sys
sys.path.insert(0, '/content/TRELLIS')
os.environ.update({
    'SPCONV_ALGO': 'native',
    'OPENCV_IO_ENABLE_OPENEXR': '1',
    'PYTORCH_CUDA_ALLOC_CONF': 'expandable_segments:True',
})

from PIL import Image
from trellis.pipelines import TrellisImageTo3DPipeline
from trellis.utils import postprocessing_utils

# โหลด model
print('📥 Loading model (microsoft/TRELLIS-image-large)...')
pipeline = TrellisImageTo3DPipeline.from_pretrained("microsoft/TRELLIS-image-large")
pipeline.cuda()
print('✅ Model loaded!')

# โหลดรูป
if not os.path.exists(IMAGE_PATH):
    raise FileNotFoundError(f'ไม่พบรูป: {IMAGE_PATH}')
image = Image.open(IMAGE_PATH).convert('RGBA')
print(f'🖼️  Image: {IMAGE_PATH} ({image.size})')

# Generate
print(f'🚀 Generating... (seed={SEED}, steps={STEPS})')
outputs = pipeline.run(
    image, seed=SEED,
    sparse_structure_sampler_params={'steps': STEPS, 'cfg_strength': 7.5},
    slat_sampler_params={'steps': STEPS, 'cfg_strength': 3.0},
)

# Export GLB
print('💾 Exporting GLB...')
glb = postprocessing_utils.to_glb(
    outputs['gaussian'][0],
    outputs['mesh'][0],
    simplify=0.95,
    texture_size=1024,
)
glb.export(OUTPUT_PATH)
print(f'✅ Saved: {OUTPUT_PATH}')

# Download
try:
    from google.colab import files
    files.download(OUTPUT_PATH)
except:
    print('💡 ดาวน์โหลดจาก Files panel ทางซ้ายมือ')

---
## 🛠️ Troubleshooting

| ปัญหา | สาเหตุ | วิธีแก้ |
|-------|--------|--------|
| Kernel crash หลัง Cell 1 | condacolab restart — ปกติ! | Run all ใหม่ |
| `conda create` ไม่ทำงาน | condacolab ใช้ base env เท่านั้น | ✅ notebook นี้แก้แล้ว |
| `CUDA out of memory` | VRAM ไม่พอ | ใช้ A100 หรือ ลด texture_size=512 |
| setup.sh ช้ามาก | compile C++ extensions | ปกติ รอ 20-30 นาที |
| Gradio URL ไม่ขึ้น | Gradio share ล่ม | กด restart + Run all |
| `ModuleNotFoundError: trellis` | path ไม่ถูก | ใส่ `sys.path.insert(0, '/content/TRELLIS')` |
| kaolin error PyTorch 2.4.0 | version mismatch | setup.sh จัดการให้อัตโนมัติ |

---
## 📚 References
- [Microsoft TRELLIS Official](https://github.com/microsoft/TRELLIS)
- [HuggingFace Model](https://huggingface.co/microsoft/TRELLIS-image-large)
- [Gradio Demo (ฟรีไม่ต้อง setup)](https://huggingface.co/spaces/Microsoft/TRELLIS)
- [jackel27/Trellis-Colab](https://github.com/jackel27/Trellis-Colab) (ต้นแบบ Community)
